![Cartoon of telecom customers](IMG_8811.png)


The telecommunications (telecom) sector in India is rapidly changing, with more and more telecom businesses being created and many customers deciding to switch between providers. "Churn" refers to the process where customers or subscribers stop using a company's services or products. Understanding the factors that influence keeping a customer as a client in predicting churn is crucial for telecom companies to enhance their service quality and customer satisfaction. As the data scientist on this project, you aim to explore the intricate dynamics of customer behavior and demographics in the Indian telecom sector in predicting customer churn, utilizing two comprehensive datasets from four major telecom partners: Airtel, Reliance Jio, Vodafone, and BSNL:

- `telecom_demographics.csv` contains information related to Indian customer demographics:

| Variable             | Description                                      |
|----------------------|--------------------------------------------------|
| `customer_id `         | Unique identifier for each customer.             |
| `telecom_partner `     | The telecom partner associated with the customer.|
| `gender `              | The gender of the customer.                      |
| `age `                 | The age of the customer.                         |
| `state`                | The Indian state in which the customer is located.|
| `city`                 | The city in which the customer is located.       |
| `pincode`              | The pincode of the customer's location.          |
| `registration_event` | When the customer registered with the telecom partner.|
| `num_dependents`      | The number of dependents (e.g., children) the customer has.|
| `estimated_salary`     | The customer's estimated salary.                 |

- `telecom_usage` contains information about the usage patterns of Indian customers:

| Variable   | Description                                                  |
|------------|--------------------------------------------------------------|
| `customer_id` | Unique identifier for each customer.                         |
| `calls_made` | The number of calls made by the customer.                    |
| `sms_sent`   | The number of SMS messages sent by the customer.             |
| `data_used`  | The amount of data used by the customer.                     |
| `churn`    | Binary variable indicating whether the customer has churned or not (1 = churned, 0 = not churned).|


In [29]:
# Import libraries and methods/functions
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

#Load csv files: telecom_demographics.csv and telecom_usage
demographics_df = pd.read_csv("telecom_demographics.csv")
usage_df = pd.read_csv("telecom_usage.csv")

#Merge demographics_df and usage_df into churn_df
churn_df = pd.merge(demographics_df, usage_df, on = 'customer_id')


In [30]:
#Calculate the proportion of customers who have churned.
churn_proportion = churn_df['churn'].mean()

print(f"This is the proportion of customers who have churned: {churn_proportion: .1%}")

This is the proportion of customers who have churned:  20.0%


In [31]:
#Identify Categorical Features
categorical_features = list(churn_df.select_dtypes(include=['object', 'category']).columns)

print(f"This is the list of categorical features: {categorical_features}")

This is the list of categorical features: ['telecom_partner', 'gender', 'state', 'city', 'registration_event']


In [32]:
#Separate Features and Target to different DataFrames
X = churn_df.drop(["customer_id","churn"], axis=1)
target = churn_df['churn']

#Encoding Categorical Features
encoded_df = pd.get_dummies(X[categorical_features], drop_first = True)

#Identify Numerical Features
numerical_features = [col for col in X.columns if col not in categorical_features]

#Scaling Numerical Features
scaler = StandardScaler()
scaled_numerical = scaler.fit_transform(X[numerical_features])

#Covert back to DataFrame
scaled_numerical_df = pd.DataFrame(
    scaled_numerical,
    columns = numerical_features,
    index = X.index
)

#Combine Numerical and Categorical Features into features_scaled
features_scaled = pd.concat([scaled_numerical_df, encoded_df], axis=1)

In [33]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    features_scaled,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target  
)

In [34]:
#Instantiate Logistic Regression Object
log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

#Train Logistic Regression Model
log_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [35]:
#Instantiate Random Forest Classifier Object
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
#Train Random Forest Classifier
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [36]:
#Logistic Regression Predictions
logreg_pred = log_model.predict(X_test)

#Random Forest predictions
rf_pred = rf_model.predict(X_test)



In [37]:
#Assess Logistic Regression Model using Confusion Matrix
cm_log = confusion_matrix(y_test, logreg_pred)

print("Logistic Regression Confusion Matrix:")
print(cm_log)

#Assess Logistic Regression Model using Classification Report
print("Logistic Regression Classification Report: ")
print(classification_report(y_test, logreg_pred))

Logistic Regression Confusion Matrix:
[[1032    7]
 [ 261    0]]
Logistic Regression Classification Report: 
              precision    recall  f1-score   support

           0       0.80      0.99      0.89      1039
           1       0.00      0.00      0.00       261

    accuracy                           0.79      1300
   macro avg       0.40      0.50      0.44      1300
weighted avg       0.64      0.79      0.71      1300



In [38]:
#Assess Random Forest Classifier using Confusing Matrix
cm_rf = confusion_matrix(y_test, rf_pred)

print("Random Forest Classifier Confusing Matrix: ")
print(cm_rf)

#Assess Random Forest Classifier using Classification Report: ")
print(classification_report(y_test, rf_pred))

Random Forest Classifier Confusing Matrix: 
[[1037    2]
 [ 261    0]]
              precision    recall  f1-score   support

           0       0.80      1.00      0.89      1039
           1       0.00      0.00      0.00       261

    accuracy                           0.80      1300
   macro avg       0.40      0.50      0.44      1300
weighted avg       0.64      0.80      0.71      1300



In [39]:
higher_accuracy = "RandomForest"